In [1]:
%cd /mlx_devbox/users/janne.spijkervet/repo/451/samantha

/mlx_devbox/users/janne.spijkervet/repo/451/samantha


In [2]:
from recipes.musiclm.datasets.mcc import WrappedMCC40MDataset

2023-08-11 00:59:17,849 - databus.databus_cache - INFO - databus python cache flush thread begin
2023-08-11 00:59:18,614 - torch.distributed.nn.jit.instantiator - INFO - Created a temporary directory at /tmp/tmp3j60todp
2023-08-11 00:59:18,616 - torch.distributed.nn.jit.instantiator - INFO - Writing /tmp/tmp3j60todp/_remote_module_non_scriptable.py


In [3]:
url2index_list = [
    "/mnt/bn/audio-diffusion/data/non_vocal_mcc_npy.filtered+audio_metrics_good/mega_index.with_ar_scores+vad/genre_specific/npy_url2idx.txt.alternative-hip-hop+chinese-style+others",
    "/mnt/bn/audio-diffusion/data/non_vocal_mcc_npy.filtered+audio_metrics_good/mega_index.with_ar_scores+vad/genre_specific/npy_url2idx.txt.blues+childhood+country+devotional+k-pop+soundtrack+trance+world-music",
    "/mnt/bn/audio-diffusion/data/non_vocal_mcc_npy.filtered+audio_metrics_good/mega_index.with_ar_scores+vad/genre_specific/npy_url2idx.txt.classical",
    "/mnt/bn/audio-diffusion/data/non_vocal_mcc_npy.filtered+audio_metrics_good/mega_index.with_ar_scores+vad/genre_specific/npy_url2idx.txt.easy-listening",
    "/mnt/bn/audio-diffusion/data/non_vocal_mcc_npy.filtered+audio_metrics_good/mega_index.with_ar_scores+vad/genre_specific/npy_url2idx.txt.electronic+techno",
    "/mnt/bn/audio-diffusion/data/non_vocal_mcc_npy.filtered+audio_metrics_good/mega_index.with_ar_scores+vad/genre_specific/npy_url2idx.txt.folk+indie-folk",
    "/mnt/bn/audio-diffusion/data/non_vocal_mcc_npy.filtered+audio_metrics_good/mega_index.with_ar_scores+vad/genre_specific/npy_url2idx.txt.hip-hop-rap",
    "/mnt/bn/audio-diffusion/data/non_vocal_mcc_npy.filtered+audio_metrics_good/mega_index.with_ar_scores+vad/genre_specific/npy_url2idx.txt.jazz",
    "/mnt/bn/audio-diffusion/data/non_vocal_mcc_npy.filtered+audio_metrics_good/mega_index.with_ar_scores+vad/genre_specific/npy_url2idx.txt.new-age",
    "/mnt/bn/audio-diffusion/data/non_vocal_mcc_npy.filtered+audio_metrics_good/mega_index.with_ar_scores+vad/genre_specific/npy_url2idx.txt.pop",
    "/mnt/bn/audio-diffusion/data/non_vocal_mcc_npy.filtered+audio_metrics_good/mega_index.with_ar_scores+vad/genre_specific/npy_url2idx.txt.rock",
]

weights = [
    1.0,
    2.0,
    4.0,
    1.0,
    2.0,
    2.0,
    2.0,
    4.0,
    2.0,
    4.0,
    2.0,
]

In [14]:
sample_rate = 24000
num_workers = 0
shuffle_buffer = 200
batch_size = 50
train_dataset = WrappedMCC40MDataset(
    url2index_list=url2index_list,
    weights=weights,
    sample_rate=sample_rate,
    duration=10,
    audio_key="audio.npy",
    min_volume_threshold=0.05,
    loudness_ratio_threshold=0.2,
    # ar_filtering: !ref <ar_filtering>
    max_num_crops=6,
    avoid_vocal=True,
    max_vocal_threshold=0.5,
    resampled=True,
    shardshuffle=True,
    use_pipe=True,
)

In [15]:
from recipes.musiclm.datamodules.webdataset import DataModule

pl_datamodule = DataModule(
    train_dataset=train_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    shuffle_buffer_size=shuffle_buffer,
    pin_memory=True,
    train_keys=["url", "metadata"],
)

In [16]:
train_loader = pl_datamodule.train_dataloader()

In [17]:
batch = next(iter(train_loader))

In [19]:
from collections import Counter

counter = Counter()

for b in batch[1]:
    counter[b["ddex_genre"]] += 1

In [20]:
counter

Counter({'Jazz / Jazz': 9,
         'Easy Listening,New Age': 8,
         'Classical': 6,
         'Pop': 4,
         'Folk / Folk': 2,
         'Alternative - Industrial': 2,
         'Ambient/New Age/Meditation,Easy Listening': 2,
         'World': 2,
         'Hip Hop/Rap': 2,
         'World Music / World Music': 2,
         'Pop - French / Variété Française': 2,
         'World Music / Celtic Folk': 2,
         'Electronica/Dance,Electronica/Dance / Electronic': 1,
         'Rock / Krautrock': 1,
         'Dance': 1,
         'Organic House / Downtempo': 1,
         'World Music / Peru': 1,
         'Pop - Dance': 1,
         'Punk': 1})